# TEMPO NO₂ event-access benchmark

This notebook implements the protocol in `docs/benchmark-no2-event.md` for a configurable TEMPO observation window. The current run targets 14 March 2025 near `36.98° N, 95.97° W` and compares remote file-based netCDF access with a pinned Virtual Zarr / Icechunk snapshot using identical observations and spatial selections.

**Prerequisites**

- Python 3.12 and Earthdata Login credentials. The notebook tries environment and netrc credentials, then prompts interactively and persists the login.
- AWS access key ID, secret access key, and session token authorized to read `s3://airquality-data-store-develop/tempo/no2/v04`. Enter them in the dedicated credential cell and never commit populated credentials.
- Run in `us-west-2` for direct access to `s3://asdc-prod-protected`.
- Cartopy downloads Natural Earth state boundaries on first use.
- Validate event-marker coordinates and sources before scientific interpretation.

The Icechunk repository uses only the explicitly supplied credential dictionary, not the JupyterHub role or global AWS credential chain. Earthdata supplies separate temporary credentials for protected ASDC source chunks.

Discovery, correctness checks, and benchmark execution are guarded by flags. No authenticated or expensive work runs until those flags are enabled.

## 1. Install and import benchmark dependencies

Uncomment the installation line only when the selected kernel is missing packages. Restart the kernel after installation.

In [ ]:
# %pip install -q "earthaccess>=0.14" cartopy h5py "icechunk>=1" matplotlib numpy pandas psutil "xarray>=2025.1" zarr obspec-utils obstore

from __future__ import annotations

import json
import platform
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Generator, Iterator, Protocol

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import earthaccess
import h5py
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import xarray as xr
from IPython.display import Image, display
from obspec_utils.readers import BlockStoreReader
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

pd.set_option("display.max_columns", 30)

## 2. Define benchmark configuration and test matrix

The point sets are nested prefixes of one seeded 250-point sample. Every AOI is a square centered at `36.98° N, 95.97° W`; `aoi_sizes_km` gives the full side length. Thus the 500 km case extends approximately 250 km north, south, east, and west from the center. Change `RUN_DISCOVERY`, `RUN_CORRECTNESS`, and `RUN_BENCHMARKS` deliberately as the workflow progresses.

In [ ]:
@dataclass(frozen=True)
class BenchmarkConfig:
    concept_id: str = "C3685896708-LARC_CLOUD"
    event_start: str = "2025-03-14T00:00:00Z"
    event_end: str = "2025-03-15T00:00:00Z"
    map_time: str = "2025-03-14T19:14:59Z"
    map_search_minutes: int = 90
    center_lat: float = 36.98
    center_lon: float = -95.97
    aoi_sizes_km: tuple[int, ...] = (50, 100, 250, 500)
    point_counts: tuple[int, ...] = (1, 10, 50, 250)
    random_seed: int = 20250314
    accepted_quality_flag: int = 0  # V04: normal=0, suspicious=1, bad=2.
    repetitions: int = 5
    warmups: int = 1
    store_bucket: str = "airquality-data-store-develop"
    store_prefix: str = "tempo/no2/v04"
    region: str = "us-west-2"
    output_dir: str = "benchmark-results/no2-2025-03-14"


CONFIG = BenchmarkConfig()
OUTPUT_DIR = Path(CONFIG.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_DISCOVERY = True
RUN_CORRECTNESS = False
RUN_BENCHMARKS = False
RUN_VISUALIZATION = True

# Add only verified event observations. Required columns are shown below.
# Example:
# EVENT_MARKERS.loc[len(EVENT_MARKERS)] = {
#     "name": "FIRMS detection", "latitude": 36.8, "longitude": -96.2,
#     "kind": "wildfire", "source": "NASA FIRMS <URL or record ID>",
# }
EVENT_MARKERS = pd.DataFrame(
    columns=["name", "latitude", "longitude", "kind", "source"]
)

TEST_MATRIX = pd.concat(
    [
        pd.DataFrame({"workload": "point", "case_size": CONFIG.point_counts}),
        pd.DataFrame({"workload": "aoi", "case_size": CONFIG.aoi_sizes_km}),
    ],
    ignore_index=True,
)
TEST_MATRIX

In [ ]:
# Credentials for the Icechunk repository bucket only.
# Replace these placeholders in your private notebook session. Do not commit secrets.
STORE_AWS_CREDENTIALS = {
    "access_key_id": "YOUR_AWS_ACCESS_KEY_ID",
    "secret_access_key": "YOUR_AWS_SECRET_ACCESS_KEY",
    "session_token": "YOUR_AWS_SESSION_TOKEN",
}

if any(value.startswith("YOUR_") for value in STORE_AWS_CREDENTIALS.values()):
    print("Set STORE_AWS_CREDENTIALS before opening the Icechunk repository.")

## 3. Prepare input data and storage targets

CMR discovery runs once and writes a pinned granule manifest. Subsequent sessions load that file. The Icechunk branch tip is also resolved once and recorded as a snapshot ID.

In [ ]:
MANIFEST_PATH = OUTPUT_DIR / "run-manifest.json"


def earthdata_login() -> Any:
    """Use environment/netrc credentials or prompt interactively on JupyterHub."""
    return earthaccess.login(strategy="all", persist=True)


def direct_s3_url(granule: Any) -> str:
    links = [url for url in granule.data_links(access="direct") if url.endswith(".nc")]
    if not links:
        raise RuntimeError(f"No direct S3 netCDF link for {granule['meta']['concept-id']}")
    return links[0]


def discover_event_granules() -> pd.DataFrame:
    earthdata_login()
    granules = earthaccess.search_data(
        concept_id=CONFIG.concept_id,
        temporal=(CONFIG.event_start, CONFIG.event_end),
        count=-1,
        sort_key="start_date",
    )
    records = []
    for granule in granules:
        umm = granule["umm"]
        temporal = umm["TemporalExtent"]["RangeDateTime"]
        archive = umm.get("DataGranule", {}).get("ArchiveAndDistributionInformation", [])
        netcdf_file = next((item for item in archive if item.get("Name", "").endswith(".nc")), {})
        records.append(
            {
                "granule_ur": umm["GranuleUR"],
                "concept_id": granule["meta"]["concept-id"],
                "revision_id": granule["meta"].get("revision-id"),
                "start_time": temporal["BeginningDateTime"],
                "end_time": temporal["EndingDateTime"],
                "source_url": direct_s3_url(granule),
                "source_size_mb": netcdf_file.get("Size"),
            }
        )
    if not records:
        raise RuntimeError("CMR returned no event-day granules")
    return pd.DataFrame(records).sort_values("start_time").reset_index(drop=True)


if RUN_DISCOVERY:
    granule_manifest = discover_event_granules()
    payload = {
        "created_at": datetime.now(UTC).isoformat(),
        "config": asdict(CONFIG),
        "environment": {"python": platform.python_version(), "platform": platform.platform()},
        "granules": granule_manifest.to_dict(orient="records"),
    }
    MANIFEST_PATH.write_text(json.dumps(payload, indent=2) + "\n")
elif MANIFEST_PATH.exists():
    payload = json.loads(MANIFEST_PATH.read_text())
    granule_manifest = pd.DataFrame(payload["granules"])
else:
    granule_manifest = pd.DataFrame()
    print(f"Set RUN_DISCOVERY=True to create {MANIFEST_PATH}")

granule_manifest

In [ ]:
SOURCE_CONTAINER = "s3://asdc-prod-protected/"


def asdc_credentials() -> dict[str, Any]:
    earthdata_login()
    credentials = earthaccess.get_s3_credentials(daac="ASDC")
    required = ("accessKeyId", "secretAccessKey", "sessionToken")
    missing = [key for key in required if not credentials.get(key)]
    if missing:
        raise RuntimeError(f"ASDC credential response missing {missing}")
    return credentials


def source_registry(credentials: dict[str, Any]) -> ObjectStoreRegistry:
    store = S3Store(
        "asdc-prod-protected",
        region=CONFIG.region,
        access_key_id=credentials["accessKeyId"],
        secret_access_key=credentials["secretAccessKey"],
        token=credentials["sessionToken"],
    )
    return ObjectStoreRegistry({SOURCE_CONTAINER.rstrip("/"): store})


def open_virtual_dataset(credentials: dict[str, Any], snapshot_id: str | None = None):
    if any(value.startswith("YOUR_") for value in STORE_AWS_CREDENTIALS.values()):
        raise RuntimeError("Replace the STORE_AWS_CREDENTIALS placeholders first")
    storage = icechunk.s3_storage(
        bucket=CONFIG.store_bucket,
        prefix=CONFIG.store_prefix,
        region=CONFIG.region,
        access_key_id=STORE_AWS_CREDENTIALS["access_key_id"],
        secret_access_key=STORE_AWS_CREDENTIALS["secret_access_key"],
        session_token=STORE_AWS_CREDENTIALS["session_token"],
    )
    repository = icechunk.Repository.open(
        storage=storage,
        authorize_virtual_chunk_access=icechunk.containers_credentials(
            {
                SOURCE_CONTAINER: icechunk.s3_credentials(
                    access_key_id=credentials["accessKeyId"],
                    secret_access_key=credentials["secretAccessKey"],
                    session_token=credentials["sessionToken"],
                )
            }
        ),
    )
    pinned_snapshot = snapshot_id or repository.lookup_branch("main")
    dataset = xr.open_dataset(
        repository.readonly_session(snapshot_id=pinned_snapshot).store,
        engine="zarr",
        consolidated=False,
        zarr_format=3,
    )
    return repository, pinned_snapshot, dataset


credentials = None
repository = None
snapshot_id = None
virtual_ds = None
if RUN_DISCOVERY:
    credentials = asdc_credentials()
    repository, snapshot_id, virtual_ds = open_virtual_dataset(credentials)
    payload = json.loads(MANIFEST_PATH.read_text())
    payload["icechunk_snapshot_id"] = snapshot_id
    payload["icechunk_uri"] = f"s3://{CONFIG.store_bucket}/{CONFIG.store_prefix}"
    MANIFEST_PATH.write_text(json.dumps(payload, indent=2) + "\n")
    print(f"Pinned Icechunk snapshot: {snapshot_id}")

In [ ]:
def aoi_bounds(size_km: float) -> tuple[float, float, float, float]:
    half = size_km / 2
    delta_lat = half / 110.574
    delta_lon = half / (111.320 * np.cos(np.deg2rad(CONFIG.center_lat)))
    return (
        CONFIG.center_lat - delta_lat,
        CONFIG.center_lat + delta_lat,
        CONFIG.center_lon - delta_lon,
        CONFIG.center_lon + delta_lon,
    )


def coordinate_slice(coordinates: np.ndarray, low: float, high: float) -> slice:
    indices = np.flatnonzero((coordinates >= low) & (coordinates <= high))
    if not len(indices):
        raise ValueError(f"No coordinates in [{low}, {high}]")
    return slice(int(indices[0]), int(indices[-1]) + 1)


def nearest_indices(coordinates: np.ndarray, requested: np.ndarray) -> np.ndarray:
    ascending = coordinates[0] <= coordinates[-1]
    values = coordinates if ascending else coordinates[::-1]
    right = np.searchsorted(values, requested).clip(1, len(values) - 1)
    left = right - 1
    chosen = np.where(
        np.abs(values[right] - requested) < np.abs(values[left] - requested),
        right,
        left,
    )
    return chosen if ascending else len(values) - 1 - chosen


def prepare_spatial_cases(dataset: xr.Dataset):
    latitudes = np.asarray(dataset["latitude"].values)
    longitudes = np.asarray(dataset["longitude"].values)
    outer = aoi_bounds(max(CONFIG.aoi_sizes_km))
    rng = np.random.default_rng(CONFIG.random_seed)
    points = pd.DataFrame(
        {
            "point_id": np.arange(max(CONFIG.point_counts)),
            "requested_lat": rng.uniform(outer[0], outer[1], max(CONFIG.point_counts)),
            "requested_lon": rng.uniform(outer[2], outer[3], max(CONFIG.point_counts)),
        }
    )
    points["lat_index"] = nearest_indices(latitudes, points["requested_lat"].to_numpy())
    points["lon_index"] = nearest_indices(longitudes, points["requested_lon"].to_numpy())
    points["pixel_lat"] = latitudes[points["lat_index"]]
    points["pixel_lon"] = longitudes[points["lon_index"]]
    windows = {
        size: (
            coordinate_slice(latitudes, *aoi_bounds(size)[:2]),
            coordinate_slice(longitudes, *aoi_bounds(size)[2:]),
        )
        for size in CONFIG.aoi_sizes_km
    }
    return latitudes, longitudes, points, windows


def manifest_matches_config(run_payload: dict[str, Any]) -> bool:
    stored = run_payload.get("config", {})
    if any(
        stored.get(key) != getattr(CONFIG, key)
        for key in ("concept_id", "event_start", "event_end")
    ):
        return False
    granules = run_payload.get("granules", [])
    if not granules:
        return False
    starts = pd.to_datetime(
        [granule["start_time"] for granule in granules], utc=True
    )
    return bool(
        (starts >= pd.Timestamp(CONFIG.event_start)).all()
        and (starts < pd.Timestamp(CONFIG.event_end)).all()
    )


def new_run_payload(manifest: pd.DataFrame) -> dict[str, Any]:
    return {
        "created_at": datetime.now(UTC).isoformat(),
        "config": asdict(CONFIG),
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
        },
        "granules": manifest.to_dict(orient="records"),
    }


def prepare_benchmark_inputs():
    run_payload = None
    if MANIFEST_PATH.exists():
        candidate = json.loads(MANIFEST_PATH.read_text())
        if manifest_matches_config(candidate):
            run_payload = candidate
        else:
            print("Cached manifest does not match this date; rediscovering granules.")

    if run_payload is None:
        manifest = discover_event_granules()
        run_payload = new_run_payload(manifest)
    else:
        manifest = pd.DataFrame(run_payload["granules"])

    source_credentials = asdc_credentials()
    repo, pinned_snapshot, dataset = open_virtual_dataset(
        source_credentials, run_payload.get("icechunk_snapshot_id")
    )
    run_payload["config"] = asdict(CONFIG)
    run_payload["icechunk_snapshot_id"] = pinned_snapshot
    run_payload["icechunk_uri"] = (
        f"s3://{CONFIG.store_bucket}/{CONFIG.store_prefix}"
    )
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(run_payload, indent=2) + "\n")
    print(f"Benchmark manifest: {MANIFEST_PATH.resolve()}")
    return run_payload, manifest, source_credentials, repo, pinned_snapshot, dataset


if RUN_CORRECTNESS or RUN_BENCHMARKS or RUN_VISUALIZATION:
    (
        payload,
        granule_manifest,
        credentials,
        repository,
        snapshot_id,
        virtual_ds,
    ) = prepare_benchmark_inputs()
    latitudes, longitudes, points, aoi_windows = prepare_spatial_cases(virtual_ds)
    display(points.head(10), pd.DataFrame.from_dict(aoi_windows, orient="index"))

## 4. Implement the baseline processing pipeline

The baseline reads each pinned netCDF granule through an authenticated ranged S3 reader. File-open cost is included in retrieval because each observation is a separate file. It never downloads a complete granule unless the HDF5 access pattern requires those ranges.

In [ ]:
NO2_PATH = "product/vertical_column_troposphere"
QUALITY_PATH = "product/main_data_quality_flag"
TEMPO_EPOCH = np.datetime64("1980-01-06T00:00:00", "ns")


@dataclass
class SelectionResult:
    times: np.ndarray
    no2: np.ndarray
    quality: np.ndarray


class BenchmarkBackend(Protocol):
    name: str

    def point(self, observations: list[int], point_rows: pd.DataFrame) -> SelectionResult: ...

    def aoi(self, observations: list[int], window: tuple[slice, slice]) -> SelectionResult: ...


def _spatial_key(dataset: h5py.Dataset, lat_key: Any, lon_key: Any) -> tuple[Any, ...]:
    if dataset.ndim < 2:
        raise ValueError(f"Expected a spatial array, got shape {dataset.shape}")
    return (0,) * (dataset.ndim - 2) + (lat_key, lon_key)


def _decode_cf(dataset: h5py.Dataset, raw: Any) -> np.ndarray:
    raw_values = np.asarray(raw)
    fill = dataset.attrs.get("_FillValue")
    mask = np.zeros(raw_values.shape, dtype=bool)
    if fill is not None:
        mask = raw_values == np.asarray(fill).reshape(-1)[0]
    values = raw_values.astype(np.float64)
    if "scale_factor" in dataset.attrs:
        values *= float(np.asarray(dataset.attrs["scale_factor"]).reshape(-1)[0])
    if "add_offset" in dataset.attrs:
        values += float(np.asarray(dataset.attrs["add_offset"]).reshape(-1)[0])
    return np.where(mask, np.nan, values)


def _source_time(handle: h5py.File) -> np.datetime64:
    seconds = float(np.asarray(handle["time"][0]))
    return TEMPO_EPOCH + np.timedelta64(round(seconds * 1_000_000_000), "ns")


class RemoteNetCDFBackend:
    name = "remote_netcdf"

    def __init__(self, manifest: pd.DataFrame, registry: ObjectStoreRegistry):
        self.manifest = manifest.reset_index(drop=True)
        self.registry = registry

    @contextmanager
    def _open(self, observation: int) -> Iterator[h5py.File]:
        url = self.manifest.iloc[observation]["source_url"]
        store, path = self.registry.resolve(url)
        with BlockStoreReader(store, path) as reader, h5py.File(reader, "r") as handle:
            yield handle

    def point(self, observations: list[int], point_rows: pd.DataFrame) -> SelectionResult:
        times, no2_rows, quality_rows = [], [], []
        for observation in observations:
            with self._open(observation) as handle:
                times.append(_source_time(handle))
                no2_dataset = handle[NO2_PATH]
                quality_dataset = handle[QUALITY_PATH]
                no2_rows.append([
                    _decode_cf(no2_dataset, no2_dataset[_spatial_key(no2_dataset, int(row.lat_index), int(row.lon_index))]).item()
                    for row in point_rows.itertuples()
                ])
                quality_rows.append([
                    _decode_cf(quality_dataset, quality_dataset[_spatial_key(quality_dataset, int(row.lat_index), int(row.lon_index))]).item()
                    for row in point_rows.itertuples()
                ])
        return SelectionResult(np.asarray(times), np.asarray(no2_rows), np.asarray(quality_rows))

    def aoi(self, observations: list[int], window: tuple[slice, slice]) -> SelectionResult:
        times, no2_rows, quality_rows = [], [], []
        for observation in observations:
            with self._open(observation) as handle:
                times.append(_source_time(handle))
                no2_dataset = handle[NO2_PATH]
                quality_dataset = handle[QUALITY_PATH]
                no2_rows.append(_decode_cf(no2_dataset, no2_dataset[_spatial_key(no2_dataset, *window)]))
                quality_rows.append(_decode_cf(quality_dataset, quality_dataset[_spatial_key(quality_dataset, *window)]))
        return SelectionResult(np.asarray(times), np.stack(no2_rows), np.stack(quality_rows))

## 5. Implement the Virtual Zarr processing pipeline

The virtual adapter maps each pinned source granule to the nearest exact store time within four minutes, then performs the same point or rectangular index selection. Calling `.load()` defines completion of retrieval.

In [ ]:
class VirtualZarrBackend:
    name = "virtual_zarr"

    def __init__(self, dataset: xr.Dataset, manifest: pd.DataFrame):
        self.dataset = dataset
        store_times = pd.DatetimeIndex(dataset["time"].values).tz_localize("UTC")
        nominal = pd.to_datetime(manifest["start_time"], utc=True)
        indexer = store_times.get_indexer(nominal, method="nearest", tolerance=pd.Timedelta("4min"))
        if np.any(indexer < 0):
            missing = manifest.loc[indexer < 0, "granule_ur"].tolist()
            raise RuntimeError(f"Event granules missing from Icechunk snapshot: {missing}")
        self.time_indices = indexer

    def _selection(self, observations: list[int], indexers: dict[str, Any]) -> SelectionResult:
        time_indices = self.time_indices[observations]
        selected = self.dataset[["vertical_column_troposphere", "main_data_quality_flag"]].isel(
            time=xr.DataArray(time_indices, dims="observation"), **indexers
        ).load()
        return SelectionResult(
            np.asarray(selected["time"].values, dtype="datetime64[ns]"),
            np.asarray(selected["vertical_column_troposphere"].values, dtype=np.float64),
            np.asarray(selected["main_data_quality_flag"].values, dtype=np.float64),
        )

    def point(self, observations: list[int], point_rows: pd.DataFrame) -> SelectionResult:
        return self._selection(
            observations,
            {
                "latitude": xr.DataArray(point_rows["lat_index"].to_numpy(), dims="point"),
                "longitude": xr.DataArray(point_rows["lon_index"].to_numpy(), dims="point"),
            },
        )

    def aoi(self, observations: list[int], window: tuple[slice, slice]) -> SelectionResult:
        return self._selection(observations, {"latitude": window[0], "longitude": window[1]})


def quality_mask(result: SelectionResult) -> np.ndarray:
    """Apply the provisional V04 rule; replace only after checking the product guide."""
    return np.where(result.quality == CONFIG.accepted_quality_flag, result.no2, np.nan)


def point_table(result: SelectionResult, point_rows: pd.DataFrame) -> pd.DataFrame:
    observations, point_count = result.no2.shape
    repeated = pd.concat([point_rows.reset_index(drop=True)] * observations, ignore_index=True)
    repeated["observation_time"] = np.repeat(result.times, point_count)
    repeated["tropospheric_no2"] = quality_mask(result).reshape(-1)
    repeated["quality_flag"] = result.quality.reshape(-1)
    return repeated


def aoi_statistics(result: SelectionResult) -> pd.DataFrame:
    values = quality_mask(result)
    axes = tuple(range(1, values.ndim))
    valid = np.isfinite(values)
    return pd.DataFrame({
        "observation_time": result.times,
        "mean_no2": np.nanmean(values, axis=axes),
        "median_no2": np.nanmedian(values, axis=axes),
        "maximum_no2": np.nanmax(values, axis=axes),
        "std_no2": np.nanstd(values, axis=axes),
        "valid_pixel_count": valid.sum(axis=axes),
        "total_pixel_count": int(np.prod(values.shape[1:])),
        "spatial_coverage": valid.mean(axis=axes),
    })

## 6. Add timing, memory, and I/O instrumentation

Timing uses `perf_counter`. A sampler records peak RSS for the process and its children. Transport bytes, request counts, and object counts remain null until instrumentation is added at the S3 client boundary; array `nbytes` is intentionally not reported as transferred data.

In [ ]:
import threading


def process_rss_bytes() -> int:
    process = psutil.Process()
    processes = [process, *process.children(recursive=True)]
    return sum(item.memory_info().rss for item in processes if item.is_running())


@contextmanager
def sample_peak_rss(interval_seconds: float = 0.01) -> Iterator[list[int]]:
    samples = [process_rss_bytes()]
    stop = threading.Event()

    def sample() -> None:
        while not stop.wait(interval_seconds):
            samples.append(process_rss_bytes())

    thread = threading.Thread(target=sample, daemon=True)
    thread.start()
    try:
        yield samples
    finally:
        stop.set()
        thread.join()
        samples.append(process_rss_bytes())


def run_trial(
    backend_factory: Any,
    workload: str,
    case_size: int,
    observations: list[int],
    repetition: int,
) -> tuple[dict[str, Any], Any]:
    total_start = time.perf_counter()
    with sample_peak_rss() as rss_samples:
        open_start = time.perf_counter()
        backend = backend_factory()
        open_seconds = time.perf_counter() - open_start

        retrieval_start = time.perf_counter()
        if workload == "point":
            selection = backend.point(observations, points.iloc[:case_size])
        else:
            selection = backend.aoi(observations, aoi_windows[case_size])
        retrieval_seconds = time.perf_counter() - retrieval_start

        processing_start = time.perf_counter()
        artifact = (
            point_table(selection, points.iloc[:case_size])
            if workload == "point"
            else aoi_statistics(selection)
        )
        processing_seconds = time.perf_counter() - processing_start

    record = {
        "method": backend.name,
        "cache_state": "fresh_backend",
        "workload": workload,
        "case_size": case_size,
        "repetition": repetition,
        "observation_count": len(observations),
        "open_seconds": open_seconds,
        "first_data_seconds": open_seconds + retrieval_seconds,
        "retrieval_seconds": retrieval_seconds,
        "processing_seconds": processing_seconds,
        "rendering_seconds": 0.0,
        "end_to_end_seconds": time.perf_counter() - total_start,
        "bytes_transferred": np.nan,
        "request_count": np.nan,
        "objects_accessed": np.nan,
        "peak_rss_bytes": max(rss_samples),
        "status": "ok",
    }
    return record, artifact

## 7. Run parameterized benchmark trials

This section defines the randomized trial runner. Actual execution follows the correctness gate in the next section. Notebook trials create a fresh backend but cannot guarantee a fresh operating-system, CDN, or object-store cache; use the same functions from separate processes for publication-quality cold trials.

In [ ]:
def backend_factories() -> dict[str, Any]:
    registry = source_registry(credentials)

    def remote_factory() -> RemoteNetCDFBackend:
        return RemoteNetCDFBackend(granule_manifest, registry)

    def virtual_factory() -> VirtualZarrBackend:
        _, _, dataset = open_virtual_dataset(credentials, snapshot_id)
        return VirtualZarrBackend(dataset, granule_manifest)

    return {"remote_netcdf": remote_factory, "virtual_zarr": virtual_factory}


def execute_benchmarks() -> pd.DataFrame:
    factories = backend_factories()
    observations = list(range(len(granule_manifest)))
    records: list[dict[str, Any]] = []
    rng = np.random.default_rng(CONFIG.random_seed)

    for case in TEST_MATRIX.itertuples(index=False):
        for _ in range(CONFIG.warmups):
            for factory in factories.values():
                run_trial(factory, case.workload, int(case.case_size), observations, -1)
        for repetition in range(CONFIG.repetitions):
            methods = list(factories)
            rng.shuffle(methods)
            for method in methods:
                try:
                    record, _ = run_trial(
                        factories[method], case.workload, int(case.case_size), observations, repetition
                    )
                except Exception as error:
                    record = {
                        "method": method,
                        "cache_state": "fresh_backend",
                        "workload": case.workload,
                        "case_size": int(case.case_size),
                        "repetition": repetition,
                        "observation_count": len(observations),
                        "status": f"{type(error).__name__}: {error}",
                    }
                records.append(record)
    return pd.DataFrame(records)

## 8. Validate result equivalence across pipelines

One observation is checked at every point count and AOI size before timed trials are accepted. The gate requires identical times, shapes, normalized dtypes, flags, NaN masks, and decoded NO₂ values.

In [ ]:
def assert_equivalent(baseline: SelectionResult, virtual: SelectionResult) -> None:
    assert np.array_equal(baseline.times, virtual.times), "Observation times differ"
    assert baseline.no2.shape == virtual.no2.shape
    assert baseline.quality.shape == virtual.quality.shape
    assert baseline.no2.dtype == virtual.no2.dtype
    assert baseline.quality.dtype == virtual.quality.dtype
    assert np.array_equal(baseline.quality, virtual.quality, equal_nan=True)
    assert np.array_equal(np.isnan(baseline.no2), np.isnan(virtual.no2))
    np.testing.assert_allclose(baseline.no2, virtual.no2, rtol=0, atol=0, equal_nan=True)
    np.testing.assert_allclose(
        quality_mask(baseline), quality_mask(virtual), rtol=0, atol=0, equal_nan=True
    )


def validate_all_cases() -> pd.DataFrame:
    factories = backend_factories()
    baseline = factories["remote_netcdf"]()
    virtual = factories["virtual_zarr"]()
    rows = []
    for count in CONFIG.point_counts:
        left = baseline.point([0], points.iloc[:count])
        right = virtual.point([0], points.iloc[:count])
        assert_equivalent(left, right)
        rows.append({"workload": "point", "case_size": count, "status": "pass"})
    for size in CONFIG.aoi_sizes_km:
        left = baseline.aoi([0], aoi_windows[size])
        right = virtual.aoi([0], aoi_windows[size])
        assert_equivalent(left, right)
        rows.append({"workload": "aoi", "case_size": size, "status": "pass"})
    return pd.DataFrame(rows)


correctness_results = pd.DataFrame()
raw_results = pd.DataFrame()
if RUN_CORRECTNESS:
    correctness_results = validate_all_cases()
    display(correctness_results)
    if RUN_BENCHMARKS:
        raw_results = execute_benchmarks()
        raw_results.to_csv(OUTPUT_DIR / "raw-metrics.csv", index=False)
        display(raw_results)
elif RUN_BENCHMARKS:
    raise RuntimeError("Set RUN_CORRECTNESS=True; benchmarks require a passing gate")

## 9. Aggregate metrics and compute comparative statistics

Summaries include mean, median, standard deviation, IQR, and an approximate 95% confidence interval for the mean. Speedups are paired by workload, case size, and repetition. Transfer reduction is computed only when measured transport bytes exist.

In [ ]:
RAW_RESULTS_PATH = OUTPUT_DIR / "raw-metrics.csv"
if raw_results.empty and RAW_RESULTS_PATH.exists():
    raw_results = pd.read_csv(RAW_RESULTS_PATH)


def aggregate_metric(frame: pd.DataFrame, metric: str) -> pd.DataFrame:
    groups = ["method", "cache_state", "workload", "case_size"]
    values = frame.loc[frame["status"] == "ok"].groupby(groups)[metric]
    summary = values.agg(["count", "mean", "median", "std", "min"]).reset_index()
    quantiles = values.quantile([0.25, 0.75]).unstack().reset_index()
    quantiles.columns = [*groups, "q25", "q75"]
    summary = summary.merge(quantiles, on=groups)
    summary["iqr"] = summary["q75"] - summary["q25"]
    margin = 1.96 * summary["std"] / np.sqrt(summary["count"])
    summary["mean_ci95_low"] = summary["mean"] - margin
    summary["mean_ci95_high"] = summary["mean"] + margin
    summary.insert(len(groups), "metric", metric)
    return summary


def paired_comparisons(frame: pd.DataFrame) -> pd.DataFrame:
    keys = ["cache_state", "workload", "case_size", "repetition"]
    valid = frame.loc[frame["status"] == "ok"]
    times = valid.pivot(index=keys, columns="method", values="end_to_end_seconds").dropna()
    comparison = times.reset_index()
    comparison["speedup"] = comparison["remote_netcdf"] / comparison["virtual_zarr"]
    if valid["bytes_transferred"].notna().all():
        byte_values = valid.pivot(index=keys, columns="method", values="bytes_transferred").dropna()
        comparison = comparison.merge(byte_values.reset_index(), on=keys, suffixes=("_seconds", "_bytes"))
        comparison["transfer_reduction"] = 1 - (
            comparison["virtual_zarr_bytes"] / comparison["remote_netcdf_bytes"]
        )
    else:
        comparison["transfer_reduction"] = np.nan
    return comparison


summary_results = pd.DataFrame()
comparison_results = pd.DataFrame()
if not raw_results.empty:
    metrics = [
        "open_seconds", "retrieval_seconds", "processing_seconds",
        "end_to_end_seconds", "peak_rss_bytes",
    ]
    summary_results = pd.concat(
        [aggregate_metric(raw_results, metric) for metric in metrics], ignore_index=True
    )
    comparison_results = paired_comparisons(raw_results)
    summary_results.to_csv(OUTPUT_DIR / "summary-metrics.csv", index=False)
    comparison_results.to_csv(OUTPUT_DIR / "paired-comparisons.csv", index=False)
    display(summary_results, comparison_results)

## 10. Visualize performance and export benchmark artifacts

The first science definition uses the outer portion of the largest configured window (500 × 500 km) as a background region and the central 100 × 100 km window as the event region. Enhanced NO₂ is background median plus three scaled MADs. Pixel areas use spherical geodesic geometry. This definition and the provisional quality rule must be reviewed before scientific interpretation.

In [ ]:
EARTH_RADIUS_M = 6_371_008.8
LARGEST_AOI_KM = max(CONFIG.aoi_sizes_km)


def coordinate_edges(values: np.ndarray) -> np.ndarray:
    midpoints = (values[:-1] + values[1:]) / 2
    return np.concatenate((
        [values[0] - (midpoints[0] - values[0])],
        midpoints,
        [values[-1] + (values[-1] - midpoints[-1])],
    ))


def pixel_areas_m2(latitude_values: np.ndarray, longitude_values: np.ndarray) -> np.ndarray:
    lat_edges = np.deg2rad(coordinate_edges(latitude_values))
    lon_edges = np.deg2rad(coordinate_edges(longitude_values))
    lat_band = np.abs(np.sin(lat_edges[1:]) - np.sin(lat_edges[:-1]))
    lon_width = np.abs(np.diff(lon_edges))
    return EARTH_RADIUS_M**2 * lat_band[:, None] * lon_width[None, :]


def hotspot_metrics(
    result: SelectionResult,
    latitude_values: np.ndarray,
    longitude_values: np.ndarray,
) -> pd.DataFrame:
    values = quality_mask(result)
    lat_grid, lon_grid = np.meshgrid(latitude_values, longitude_values, indexing="ij")
    inner = aoi_bounds(100)
    event_mask = (
        (lat_grid >= inner[0]) & (lat_grid <= inner[1])
        & (lon_grid >= inner[2]) & (lon_grid <= inner[3])
    )
    background_mask = ~event_mask
    areas = pixel_areas_m2(latitude_values, longitude_values)
    rows = []
    for observation, field in zip(result.times, values, strict=True):
        background = field[background_mask & np.isfinite(field)]
        median = float(np.median(background))
        mad = float(np.median(np.abs(background - median)))
        threshold = median + 3 * 1.4826 * mad
        enhanced = event_mask & np.isfinite(field) & (field > threshold)
        enhanced_area = float(areas[enhanced].sum())
        if enhanced_area:
            centroid_lat = float(np.average(lat_grid[enhanced], weights=areas[enhanced]))
            centroid_lon = float(np.average(lon_grid[enhanced], weights=areas[enhanced]))
        else:
            centroid_lat = centroid_lon = np.nan
        rows.append({
            "observation_time": observation,
            "background_median": median,
            "threshold": threshold,
            "maximum_no2": float(np.nanmax(np.where(event_mask, field, np.nan))),
            "enhanced_pixel_count": int(enhanced.sum()),
            "enhanced_area_km2": enhanced_area / 1_000_000,
            "enhanced_centroid_lat": centroid_lat,
            "enhanced_centroid_lon": centroid_lon,
        })
    return pd.DataFrame(rows)


hotspot_results = pd.DataFrame()
if RUN_BENCHMARKS and not raw_results.empty:
    backend = backend_factories()["virtual_zarr"]()
    selection = backend.aoi(
        list(range(len(granule_manifest))), aoi_windows[LARGEST_AOI_KM]
    )
    lat_slice, lon_slice = aoi_windows[LARGEST_AOI_KM]
    hotspot_results = hotspot_metrics(
        selection, latitudes[lat_slice], longitudes[lon_slice]
    )
    hotspot_results.to_csv(OUTPUT_DIR / "hotspot-metrics.csv", index=False)
    display(hotspot_results)

In [ ]:
def mapped_observation_times(backend: VirtualZarrBackend) -> pd.DatetimeIndex:
    values = backend.dataset["time"].isel(time=backend.time_indices).values
    return pd.DatetimeIndex(pd.to_datetime(values, utc=True))


def select_map_observation(
    backend: VirtualZarrBackend,
) -> tuple[int, SelectionResult]:
    target = pd.Timestamp(CONFIG.map_time)
    if target.tzinfo is None:
        target = target.tz_localize("UTC")
    times = mapped_observation_times(backend)
    offsets = pd.Series(np.abs(times - target))

    for observation in np.argsort(offsets.to_numpy()):
        offset = offsets.iloc[observation]
        if offset > pd.Timedelta(minutes=CONFIG.map_search_minutes):
            break
        selection = backend.aoi(
            [int(observation)], aoi_windows[LARGEST_AOI_KM]
        )
        raw_valid = int(np.isfinite(selection.no2[0]).sum())
        normal_valid = int(np.isfinite(quality_mask(selection)[0]).sum())
        if raw_valid:
            actual = pd.Timestamp(selection.times[0], tz="UTC")
            print(
                f"Map target: {target.isoformat()} | selected: {actual.isoformat()} | "
                f"offset: {offset} | raw pixels: {raw_valid:,} | "
                f"normal-quality pixels: {normal_valid:,}"
            )
            return int(observation), selection

    raise RuntimeError(
        f"No scan with finite NO₂ coverage over the {LARGEST_AOI_KM} km AOI "
        f"within {CONFIG.map_search_minutes} minutes of {CONFIG.map_time}. "
        "Rerun discovery if the manifest was created for another date."
    )


def map_values(result: SelectionResult) -> tuple[np.ndarray, str]:
    filtered = quality_mask(result)[0]
    quality = result.quality[0]
    finite_flags = quality[np.isfinite(quality)].astype(int)
    flags, counts = np.unique(finite_flags, return_counts=True)
    flag_counts = dict(zip(flags.tolist(), counts.tolist(), strict=True))
    print(f"Quality-flag counts in AOI: {flag_counts}")
    if np.isfinite(filtered).any():
        return filtered, "normal quality (flag = 0)"

    raw = result.no2[0]
    if np.isfinite(raw).any():
        print(
            "Warning: no normal-quality pixels; rendering finite NO₂ without "
            "the quality filter. Benchmark statistics remain quality-filtered."
        )
        return raw, "unfiltered; no flag-0 pixels"
    raise RuntimeError("Selected scan contains no finite NO₂ pixels in the AOI")


def render_no2_map(
    values: np.ndarray,
    latitude_values: np.ndarray,
    longitude_values: np.ndarray,
    path: Path,
    title: str,
    limits: tuple[float, float],
    quality_label: str,
) -> None:
    projection = ccrs.PlateCarree()
    figure, axis = plt.subplots(
        figsize=(11, 8),
        constrained_layout=True,
        subplot_kw={"projection": projection},
    )
    axis.set_extent(
        [
            float(longitude_values.min()),
            float(longitude_values.max()),
            float(latitude_values.min()),
            float(latitude_values.max()),
        ],
        crs=projection,
    )
    axis.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#f4f3ef")
    axis.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="#dceef5")
    axis.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.7)
    axis.add_feature(
        cfeature.STATES.with_scale("50m"),
        edgecolor="#4a5560",
        facecolor="none",
        linewidth=0.8,
    )

    image = axis.pcolormesh(
        longitude_values,
        latitude_values,
        values,
        transform=projection,
        shading="auto",
        cmap="YlOrRd",
        vmin=limits[0],
        vmax=limits[1],
        zorder=2,
    )
    gridlines = axis.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="#667078",
        alpha=0.5,
        linestyle=":",
    )
    gridlines.top_labels = False
    gridlines.right_labels = False

    axis.scatter(
        CONFIG.center_lon,
        CONFIG.center_lat,
        transform=projection,
        marker="*",
        s=130,
        color="#1261a0",
        edgecolor="white",
        linewidth=0.8,
        label="Analysis center",
        zorder=6,
    )
    if np.isfinite(values).any():
        max_row, max_column = np.unravel_index(
            np.nanargmax(values), values.shape
        )
        axis.scatter(
            longitude_values[max_column],
            latitude_values[max_row],
            transform=projection,
            marker="X",
            s=80,
            color="#111111",
            edgecolor="white",
            linewidth=0.7,
            label="Maximum NO₂",
            zorder=6,
        )

    marker_styles = {
        "wildfire": {"marker": "^", "color": "#d7301f", "label": "Wildfire"},
        "dust": {"marker": "s", "color": "#8c6d31", "label": "Dust observation"},
    }
    for marker in EVENT_MARKERS.itertuples(index=False):
        style = marker_styles.get(
            str(marker.kind).lower(),
            {"marker": "o", "color": "#6a3d9a", "label": "Event observation"},
        )
        axis.scatter(
            marker.longitude,
            marker.latitude,
            transform=projection,
            marker=style["marker"],
            s=70,
            color=style["color"],
            edgecolor="white",
            linewidth=0.7,
            label=style["label"],
            zorder=7,
        )
        axis.annotate(
            marker.name,
            (marker.longitude, marker.latitude),
            xytext=(5, 5),
            textcoords="offset points",
            transform=projection,
            fontsize=8,
            zorder=8,
        )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        pad=0.03,
        shrink=0.86,
        label="Tropospheric NO₂ (molecules/cm²)",
    )
    colorbar.formatter.set_powerlimits((0, 0))
    colorbar.update_ticks()
    handles, labels = axis.get_legend_handles_labels()
    unique = dict(zip(labels, handles, strict=True))
    axis.legend(unique.values(), unique.keys(), loc="lower left", framealpha=0.92)
    axis.set_title(f"{title}\n{LARGEST_AOI_KM} × {LARGEST_AOI_KM} km; {quality_label}")
    figure.savefig(path, dpi=170)
    plt.close(figure)


def visualization_trial(
    backend_factory: Any,
    method: str,
    limits: tuple[float, float],
    observation: int,
) -> dict[str, Any]:
    total_start = time.perf_counter()
    with sample_peak_rss() as rss_samples:
        open_start = time.perf_counter()
        backend = backend_factory()
        open_seconds = time.perf_counter() - open_start
        retrieval_start = time.perf_counter()
        selection = backend.aoi(
            [observation], aoi_windows[LARGEST_AOI_KM]
        )
        retrieval_seconds = time.perf_counter() - retrieval_start
        processing_start = time.perf_counter()
        values, quality_label = map_values(selection)
        processing_seconds = time.perf_counter() - processing_start
        rendering_start = time.perf_counter()
        lat_slice, lon_slice = aoi_windows[LARGEST_AOI_KM]
        map_path = OUTPUT_DIR / f"no2-map-{method}-{CONFIG.map_time[:10]}.png"
        render_no2_map(
            values,
            latitudes[lat_slice],
            longitudes[lon_slice],
            map_path,
            f"TEMPO tropospheric NO₂ — {selection.times[0]}",
            limits,
            quality_label,
        )
        rendering_seconds = time.perf_counter() - rendering_start
    display(Image(filename=str(map_path)))
    return {
        "method": method,
        "workload": "single_observation_map",
        "case_size": LARGEST_AOI_KM,
        "requested_time": CONFIG.map_time,
        "observation_time": selection.times[0],
        "map_path": str(map_path.resolve()),
        "open_seconds": open_seconds,
        "retrieval_seconds": retrieval_seconds,
        "processing_seconds": processing_seconds,
        "rendering_seconds": rendering_seconds,
        "time_to_first_map_seconds": time.perf_counter() - total_start,
        "bytes_transferred": np.nan,
        "request_count": np.nan,
        "peak_rss_bytes": max(rss_samples),
    }


visualization_results = pd.DataFrame()
if RUN_VISUALIZATION:
    factories = backend_factories()
    reference_backend = factories["virtual_zarr"]()
    map_observation, reference = select_map_observation(reference_backend)
    valid_reference, _ = map_values(reference)
    limits = tuple(np.nanpercentile(valid_reference, [2, 98]))
    if not np.isfinite(limits).all() or limits[0] == limits[1]:
        raise RuntimeError(f"Cannot derive finite map color limits: {limits}")
    visualization_results = pd.DataFrame([
        visualization_trial(factory, method, limits, map_observation)
        for method, factory in factories.items()
    ])
    visualization_results.to_csv(
        OUTPUT_DIR / "visualization-metrics.csv", index=False
    )
    display(visualization_results)

In [ ]:
def save_performance_figures(
    summary: pd.DataFrame,
    comparisons: pd.DataFrame,
) -> list[Path]:
    paths = []
    for metric, filename, ylabel, scale in (
        ("end_to_end_seconds", "latency.png", "Median end-to-end time (s)", 1),
        ("peak_rss_bytes", "peak-memory.png", "Median peak RSS (GB)", 1_000_000_000),
    ):
        data = summary.loc[summary["metric"] == metric]
        if data.empty:
            continue
        figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
        for axis, workload in zip(axes, ("point", "aoi"), strict=True):
            subset = data.loc[data["workload"] == workload]
            for method, group in subset.groupby("method"):
                axis.plot(
                    group["case_size"],
                    group["median"] / scale,
                    marker="o",
                    label=method.replace("_", " ").title(),
                )
            axis.set(
                xlabel="Points" if workload == "point" else "AOI side (km)",
                ylabel=ylabel,
                title=workload,
            )
            axis.legend()
        path = OUTPUT_DIR / filename
        figure.savefig(path, dpi=150)
        plt.close(figure)
        paths.append(path)

    if not comparisons.empty:
        figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
        speedup = (
            comparisons.groupby(["workload", "case_size"])["speedup"]
            .median()
            .reset_index()
        )
        for axis, workload in zip(axes, ("point", "aoi"), strict=True):
            subset = speedup.loc[speedup["workload"] == workload]
            axis.plot(
                subset["case_size"],
                subset["speedup"],
                marker="o",
                label="Virtual Zarr speedup",
            )
            axis.axhline(
                1,
                color="black",
                linewidth=1,
                label="Parity (1×)",
            )
            axis.set(
                xlabel="Points" if workload == "point" else "AOI side (km)",
                ylabel="Median paired speedup",
                title=workload,
            )
            axis.legend()
        path = OUTPUT_DIR / "speedup.png"
        figure.savefig(path, dpi=150)
        plt.close(figure)
        paths.append(path)

    if not raw_results.empty and raw_results["bytes_transferred"].notna().all():
        throughput = raw_results.assign(
            throughput_mb_s=lambda frame: (
                frame["bytes_transferred"]
                / frame["retrieval_seconds"]
                / 1_000_000
            )
        )
        throughput.to_csv(OUTPUT_DIR / "throughput-metrics.csv", index=False)
    return paths


figure_paths = []
if not summary_results.empty:
    figure_paths = save_performance_figures(summary_results, comparison_results)
    print("Saved:", *figure_paths, sep="\n  ")

### Interpretation and limitations

- `main_data_quality_flag == 0` is provisional until checked against the V04 product guide.
- The 100 × 100 km event region and surrounding portion of the 500 × 500 km window are an initial analysis definition, not an attribution result. Run threshold and background-region sensitivity tests.
- Notebook “fresh backend” trials do not clear operating-system, proxy, CDN, or object-store caches. Publication-quality cold trials should launch one fresh process per row.
- `bytes_transferred`, request count, object count, throughput, and transfer reduction remain unavailable until the underlying S3/HTTP clients expose measured transport counters.
- Remote netCDF open work occurs once per granule during retrieval; Icechunk’s dataset open is reported separately. Interpret phase-level comparisons with that storage-layout difference in mind.
- Run in `us-west-2`, randomize method order, retain failures and retries, and publish the pinned manifest, raw rows, package lock, snapshot ID, summary tables, and maps.